# Creation of Research Object with their associated Claims

## Install and load packages

In [17]:
# uncomment to install packages
# pip install rohub

In [18]:
import requests

from pathlib import Path
import json
import rohub
import re

print("All libraries imported successfully!")

All libraries imported successfully!


In [19]:
print(f"ROHub version: {rohub.version()}")

ROHub version: You are currently using package v1.2.1.


## Authenticate to Rohub (needed to create Research Objects)

Create a JSON config file (for instance in your HOME folder) if you do not have it yet.

Create a file, for example:
```
rohub_credentials.json
```

where:

```
{
  "username": "your_rohub_username",
  "password": "your_rohub_password"
}
```

Load credentials from JSON in Python

In [22]:
rohub_credentials_filename = Path.home() / "rohub_credentials.json"

with open(rohub_credentials_filename) as f:
    creds = json.load(f)

rohub.login(
    username=creds["username"],
    password=creds["password"]
)

Logged successfully as annef@simula.no.


## Get title and abstract from DOI

### Define functions to get metadata from crossref or datacite

In [23]:
def clean_jats(text):
    return re.sub(r"<.*?>", "", text)
    
def get_metadata_from_crossref(doi):
    url = f"https://api.crossref.org/works/{doi}"
    r = requests.get(url, headers={"Accept": "application/json"})
    if r.status_code != 200:
        return None

    data = r.json()["message"]
    title = data.get("title", [None])[0]
    abstract = data.get("abstract")  # often None

    return {
        "source": "Crossref",
        "title": title,
        "abstract": clean_jats(abstract)
    }


def get_metadata_from_datacite(doi):
    url = f"https://api.datacite.org/dois/{doi}"
    r = requests.get(url, headers={"Accept": "application/json"})
    if r.status_code != 200:
        return None

    data = r.json()["data"]["attributes"]
    title = data.get("titles", [{}])[0].get("title")

    descriptions = data.get("descriptions", [])
    abstract = None
    for d in descriptions:
        if d.get("descriptionType") == "Abstract":
            abstract = d.get("description")
            break

    return {
        "source": "DataCite",
        "title": title,
        "abstract": clean_jats(abstract)
    }

def get_paper_metadata(doi):
    # Try Crossref first
    result = get_metadata_from_crossref(doi)
    if result:
        return result

    # Fallback to DataCite
    return get_metadata_from_datacite(doi)

In [24]:
doi = "10.3390/urbansci9020038"
metadata = get_paper_metadata(doi)

if metadata:
    print(f"Source: {metadata['source']}")
    print(f"Title: {metadata['title']}")
    print(f"Abstract: {metadata['abstract']}")
else:
    print("No metadata found.")

Source: Crossref
Title: Sustainable Placemaking and Thermal Comfort Conditions in Urban Spaces: The Case Study of Avenida dos Aliados and Praça da Liberdade (Porto, Portugal)
Abstract: The urban microclimate of Avenida dos Aliados and Praça da Liberdade was subjected to comprehensive examination through twelve measurement campaigns at six strategic observation points over the course of two seasons, namely summer and winter, between 2019 and 2020. The study employed an objective approach based on measurements to evaluate key microclimatic factors, including air temperature, which ranged from 15 °C in winter to a peak of 38 °C in summer, and Relative Humidity (RH), which varied from 50% to 85%. Additionally, wind speed was recorded between 1.0 m/s and 2.5 m/s, along with solar radiation levels, which significantly impacted Surface Temperatures (Tsurf), reaching up to 38.0 °C in some areas. A parallel subjective survey questionnaire was conducted with 123 participants. In particular, the 

## Create a new bibliographical Research Object in Rohub

In [27]:
ro_ros_type = "Bibliography-centric Research Object"
ro_research_areas = ["Earth sciences"]

CREATE_RO = False


In [28]:
if metadata and CREATE_RO:
    ro_id = rohub.ros_create(title=metadata['title'],
                          research_areas=ro_research_areas,
                          description=metadata['abstract'],
                          ros_type=ro_ros_type,
                          use_template=True)
    print("RO id = ", ro_id)
elif metadata:
    ro=rohub.ros_load(identifier=ro_id)

Research Object was successfully created with id = 7c0160bc-7413-476b-b21e-cb596023b266
RO id =  Research Object with ID: 7c0160bc-7413-476b-b21e-cb596023b266
